# NLP · Занятие 5. Векторизация и классификация текстов

как превратить текст в числа и что при этом потеряется?

## 0. Подготовка

In [46]:
# %pip install numpy pandas scikit-learn nltk

In [47]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

TOKEN_PATTERN = r"(?u)\b[а-яё]+\b" # последовательность русских букв, без пунктуации

## 1. Мешок слов: от текста к таблице

Документом может быть отзыв, письмо или целая книга. Сейчас документ — одна короткая строка.
Словарь задаёт **общие столбцы**: одна и та же позиция во всех векторах означает одно и то же слово.

In [48]:
texts = [
    "студент читает книгу",
    "студент читает читает",
    "преподаватель читает письмо",
]
pd.Series(texts, name="документ")

,документ
0,студент читает книгу
1,студент читает читает
2,преподаватель читает письмо


### Стоп 1.

In [49]:
tokens = [text.split() for text in texts]
vocabulary = sorted({word for row in tokens for word in row}) # сортируем в алфавитном порядке
counts = [Counter(row) for row in tokens]
manual_bow = [[row[word] for word in vocabulary] for row in counts]
pd.DataFrame(manual_bow, columns=vocabulary) # таблица

,книгу,письмо,преподаватель,студент,читает
0,1,0,0,1,1
1,0,0,0,1,2
2,0,1,1,0,1


<details>
<summary>После обсуждения: пояснение</summary>

Размер матрицы — **3 × 5**. Столбцы: `книгу, письмо, преподаватель, студент, читает`.
Вторая строка — `[0, 0, 0, 1, 2]`.
Ноль означает отсутствие слова в документе, а не отсутствие документа.

</details>

### То же действие через CountVectorizer

`fit` строит словарь; `transform` превращает тексты в матрицу по этому словарю.
`fit_transform` совмещает оба шага. Пока исследуем игрушечный корпус без задачи предсказания.

In [50]:
bow = CountVectorizer(token_pattern=TOKEN_PATTERN)
X_bow = bow.fit_transform(texts)
words = bow.get_feature_names_out()
pd.DataFrame(X_bow.toarray(), columns=words)

,книгу,письмо,преподаватель,студент,читает
0,1,0,0,1,1
1,0,0,0,1,2
2,0,1,1,0,1


In [51]:
print("Форма:", X_bow.shape)
print("Ненулевых элементов:", X_bow.nnz)
print("Доля нулей:", round(1 - X_bow.nnz / np.prod(X_bow.shape), 2))

Форма: (3, 5)
Ненулевых элементов: 8
Доля нулей: 0.47


Матрица **разреженная**: библиотека хранит ненулевые значения и их позиции.
`toarray()` удобно для показа маленькой таблицы. Большую матрицу корпуса целиком
в плотную таблицу не переводим: она может занять слишком много памяти.

In [52]:
binary_bow = CountVectorizer(token_pattern=TOKEN_PATTERN, binary=True)
X_binary = binary_bow.fit_transform(texts)
pd.DataFrame(X_binary.toarray(), columns=binary_bow.get_feature_names_out())

,книгу,письмо,преподаватель,студент,читает
0,1,0,0,1,1
1,0,0,0,1,1
2,0,1,1,0,1


**Частоты:** сколько раз встретилось слово. **Бинарные признаки:** встретилось ли оно вообще.
В бинарной таблице повтор `читает` больше не виден.

### Стоп 2.

Сравните «студент читает книгу» и «книгу читает студент».
Различит ли их мешок слов? А что он сделает с документом «жираф»?
Нужно ли строить новый словарь для каждого нового документа?

In [53]:
new_texts = ["студент читает книгу", "книгу читает студент", "жираф"]
X_new = bow.transform(new_texts)
pd.DataFrame(X_new.toarray(), index=new_texts, columns=words)

,книгу,письмо,преподаватель,студент,читает
студент читает книгу,1,0,0,1,1
книгу читает студент,1,0,0,1,1
жираф,0,0,0,0,0


<details>
<summary>После обсуждения: пояснение</summary>

Первые две строки совпадают: мешок слов теряет порядок.
`жираф` отсутствует в обученном словаре и даёт нулевой вектор.
Новый текст преобразуем через **transform того же векторайзера**.
Иначе число и смысл столбцов могут измениться.

</details>

## 2. N-граммы: добавляем локальный порядок · 10 минут

Униграмма — один токен, биграмма — два соседних токена.

In [54]:
order_texts = ["такси обгоняет метро", "метро обгоняет такси"]
unigrams = CountVectorizer(token_pattern=TOKEN_PATTERN)
X_uni = unigrams.fit_transform(order_texts)
pd.DataFrame(X_uni.toarray(), columns=unigrams.get_feature_names_out())

,метро,обгоняет,такси
0,1,1,1
1,1,1,1


### Стоп 3.

Униграммы дали одинаковые строки, хотя в одном тексте обгоняет такси, а в другом — метро.
Выпишите биграммы для каждой фразы.
Какие новые столбцы позволят различить строки?
Означает ли добавление биграмм, что модель теперь понимает синтаксис?

In [55]:
bigrams = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2))
X_bigrams = bigrams.fit_transform(order_texts)
pd.DataFrame(X_bigrams.toarray(), columns=bigrams.get_feature_names_out())

,метро,метро обгоняет,обгоняет,обгоняет метро,обгоняет такси,такси,такси обгоняет
0,1,0,1,1,0,1,1
1,1,1,1,0,1,1,0


<details>
<summary>После обсуждения: пояснение</summary>

Появились `такси обгоняет`, `обгоняет метро`, `метро обгоняет`, `обгоняет такси`.
Строки различаются. Но мы только посчитали соседние пары:
грамматические роли и связи на большом расстоянии алгоритм не установил.
Больше n-грамм — больше признаков; редким сочетаниям может не хватить обучающих примеров.

</details>

### Отрицание и стоп-слова

Нормализация зависит от задачи. Для тональности нельзя бездумно удалять `не`.
Убедимся в этом на примере, где удаление одного слова делает противоположные отзывы одинаковыми.

In [56]:
negation_texts = ["фильм понравился", "фильм не понравился"]
without_ne = CountVectorizer(token_pattern=TOKEN_PATTERN, stop_words=["не"])
X_without_ne = without_ne.fit_transform(negation_texts)
pd.DataFrame(X_without_ne.toarray(), columns=without_ne.get_feature_names_out())

,понравился,фильм
0,1,1
1,1,1


In [57]:
with_ne = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2))
X_with_ne = with_ne.fit_transform(negation_texts)
pd.DataFrame(X_with_ne.toarray(), columns=with_ne.get_feature_names_out())

,не,не понравился,понравился,фильм,фильм не,фильм понравился
0,0,0,1,1,0,1
1,1,1,1,1,1,0


### Символьные n-граммы

Признаком может быть и фрагмент слова. Общие фрагменты иногда помогают со словоформами
и опечатками, но не заменяют морфологический анализ.
`char_wb` строит n-граммы внутри слов с пробелами по краям.

In [58]:
char_vectorizer = CountVectorizer(analyzer="char_wb", ngram_range=(3, 3))
X_char = char_vectorizer.fit_transform(["книга", "книги"])
pd.DataFrame(X_char.toarray(), index=["книга", "книги"],
             columns=char_vectorizer.get_feature_names_out())

,кн,га,ги,ига,иги,кни,ниг
книга,1,1,0,1,0,1,1
книги,1,0,1,0,1,1,1


## 3. TF-IDF: от количества к весу · 15 минут

Частое во всей коллекции слово обычно хуже различает документы.
TF отвечает за частоту **в одном документе**, IDF — за распространённость **по документам корпуса**.

Есть несколько вариантов формулы. В этом ноутбуке считаем именно вариант
`TfidfVectorizer` с настройками по умолчанию:

$$\mathrm{tf}(t,d)=\mathrm{count}(t,d),\qquad
\mathrm{idf}(t)=1+\ln\frac{1+N}{1+\mathrm{df}(t)}.$$

$N$ — число документов, $\mathrm{df}(t)$ — число документов, где слово встретилось хотя бы раз.
Сначала $w_{t,d}=\mathrm{tf}(t,d)\cdot\mathrm{idf}(t)$,
затем каждая ненулевая строка делится на свою L2-норму:

$$\widehat w_d=\frac{w_d}{\sqrt{\sum_t w_{t,d}^2}}.$$


### Стоп 4.

Вернитесь к трём документам раздела 1.
Для `читает` чему равно df: 3 или 4? Какой IDF больше: у `читает` или у `письмо`?
Получит ли слово из всех документов IDF = 0 **по нашей формуле**?

In [59]:
N = X_bow.shape[0]
df = np.asarray((X_bow > 0).sum(axis=0)).ravel()
manual_idf = 1 + np.log((1 + N) / (1 + df))
pd.DataFrame({"слово": words, "df": df, "idf": manual_idf.round(3)})

,слово,df,idf
0,книгу,1,1.693
1,письмо,1,1.693
2,преподаватель,1,1.693
3,студент,2,1.288
4,читает,3,1.000


<details>
<summary>После обсуждения: пояснение</summary>

`df(читает) = 3`: повтор внутри второго документа не увеличивает число документов.
Для `читает` IDF = 1, для `письмо` — примерно 1.693.
**В этом варианте TF-IDF слова не обнуляются автоматически**, даже если есть во всех документах.
Стоп-слова и параметры частотных отсечений выбирают под задачу отдельно.

</details>

In [60]:
manual_weights = X_bow.toarray() * manual_idf
pd.DataFrame(manual_weights.round(3), columns=words)

,книгу,письмо,преподаватель,студент,читает
0,1.693,0.000,0.000,1.288,1.0
1,0.000,0.000,0.000,1.288,2.0
2,0.000,1.693,1.693,0.000,1.0


In [61]:
row_norms = np.linalg.norm(manual_weights, axis=1, keepdims=True)
manual_tfidf = manual_weights / row_norms
pd.DataFrame(manual_tfidf.round(3), columns=words)

,книгу,письмо,преподаватель,студент,читает
0,0.72,0.000,0.000,0.548,0.425
1,0.00,0.000,0.000,0.541,0.841
2,0.00,0.652,0.652,0.000,0.385


In [62]:
tfidf = TfidfVectorizer(token_pattern=TOKEN_PATTERN)
X_tfidf = tfidf.fit_transform(texts)
pd.DataFrame(X_tfidf.toarray().round(3), columns=tfidf.get_feature_names_out())

,книгу,письмо,преподаватель,студент,читает
0,0.72,0.000,0.000,0.548,0.425
1,0.00,0.000,0.000,0.541,0.841
2,0.00,0.652,0.652,0.000,0.385


Три представления одного корпуса: **наличие → число вхождений → вес**.
TF-IDF меняет веса признаков, но по-прежнему не устанавливает значение слов или порядок униграмм.

Обратите внимание: «редкое» не всегда значит «полезное». Опечатка тоже может получить большой IDF.

## 4. Документы как векторы: косинусная близость

Косинус сравнивает направление двух ненулевых векторов:

$$\cos(a,b)=\frac{a\cdot b}{\|a\|\,\|b\|}.$$

Для неотрицательных TF-IDF-векторов значения лежат от 0 до 1.
Если пропорционально повторить все слова, направление сохранится.

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

query_vector = tfidf.transform(["студент читает"])
scores = cosine_similarity(query_vector, X_tfidf)[0]
pd.DataFrame({"документ": texts, "близость": scores.round(3)})

,документ,близость
0,студент читает книгу,0.694
1,студент читает читает,0.943
2,преподаватель читает письмо,0.236


### Стоп 5.

Какую близость даст запрос «жираф»? Доказывает ли ноль, что тексты не связаны по смыслу?
Какую проблему создадут синонимы «автомобиль» и «машина»?

In [64]:
unknown_vector = tfidf.transform(["жираф"])
print("Ненулевых признаков:", unknown_vector.nnz)
print("Близость:", cosine_similarity(unknown_vector, X_tfidf)[0])

Ненулевых признаков: 0
Близость: [0. 0. 0.]


<details>
<summary>После обсуждения: пояснение</summary>

Получим нули: у запроса нет признаков в нашем словаре.
Косинус с нулевым вектором математически не определён; `cosine_similarity` здесь возвращает 0.
Это техническое соглашение, а не доказательство отсутствия смысловой связи.
Разные слова-синонимы остаются разными столбцами — это ограничение лексических признаков.

</details>

## 5. От векторов к классификации: тональность

**Векторайзер** получает признаки из текста. **Классификатор** учится по признакам и меткам
предсказывать класс.

Ниже — **32 вручную придуманных отзыва**, по 16 каждого класса.

Результат на них не оценивает готовность модели к реальным отзывам.
`pos` — положительный отзыв, `neg` — отрицательный.

In [65]:
positive = [
    "отличный фильм интересный сюжет", "отличный фильм актёры играют прекрасно",
    "фильм понравился буду смотреть снова", "интересный фильм и хороший финал",
    "отличный фильм рекомендую", "хороший фильм отличная музыка",
    "актёры играют прекрасно фильм понравился", "интересный сюжет и хорошие герои",
]

In [66]:
positive += [
    "фильм понравился интересная история", "отличный фильм смешные сцены",
    "хороший фильм хочется пересмотреть", "интересный фильм о дружбе",
    "финал понравился фильм отличный", "фильм понравился музыка прекрасная",
    "не скучный фильм смотрел с удовольствием", "не пожалел времени рекомендую фильм",
]

In [67]:
negative = [
    "ужасный фильм скучный сюжет", "ужасный фильм актёры играют плохо",
    "фильм не понравился зря смотрел", "скучный фильм и плохой финал",
    "ужасный фильм не рекомендую", "плохой фильм ужасная музыка",
    "актёры играют плохо фильм скучный", "скучный сюжет и неинтересные герои",
]

In [68]:
negative += [
    "фильм не понравился скучная история", "ужасный фильм несмешные сцены",
    "скучный фильм больше не посмотрю", "плохой фильм без смысла",
    "финал разочаровал фильм ужасный", "скучный фильм музыка плохая",
    "не интересный фильм жаль времени", "пожалел времени не рекомендую фильм",
]

In [69]:
reviews = pd.DataFrame({
    "text": positive + negative,
    "label": ["pos"] * len(positive) + ["neg"] * len(negative),
})
reviews["label"].value_counts()

,count
label,
pos,16
neg,16


###  разделяем тексты, обучаем векторайзер

Обучающая часть — для словаря, IDF и параметров классификатора.
Отложенная часть — для проверки заранее выбранного конвейера.
`stratify` сохраняет доли классов; `random_state` делает разбиение воспроизводимым.

In [70]:
from sklearn.model_selection import train_test_split

train_text, valid_text, train_y, valid_y = train_test_split(
    reviews["text"], reviews["label"], test_size=0.25,
    stratify=reviews["label"], random_state=42,
)
print("Обучение:", len(train_text), "Проверка:", len(valid_text))

Обучение: 24 Проверка: 8


In [71]:
# Словарь и IDF обучаем только на обучающих текстах.
review_vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN)
X_train = review_vectorizer.fit_transform(train_text)
X_valid = review_vectorizer.transform(valid_text)
print("Обучение:", X_train.shape, "Проверка:", X_valid.shape)

Обучение: (24, 44) Проверка: (8, 44)


### Наивный Байес

Классификатор сопоставляет признаки с классами. «Наивный» — про предположение
об условной независимости признаков при заданном классе, а не про простоту языка.
`MultinomialNB` часто используют как быстрый baseline для неотрицательных текстовых признаков.
TF-IDF-веса здесь допустимы, хотя это уже не буквальные целочисленные частоты событий.

In [72]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

classifier = MultinomialNB()
classifier.fit(X_train, train_y)
predicted = classifier.predict(X_valid)
print("Модель accuracy:", accuracy_score(valid_y, predicted))

Модель accuracy: 0.875


### Pipeline объединяет шаги

При `fit` конвейер обучает векторайзер, затем классификатор.
При `predict` преобразует новый текст **уже обученным** векторайзером.

In [73]:
from sklearn.pipeline import Pipeline

sentiment = Pipeline([
    ("vectorizer", TfidfVectorizer(token_pattern=TOKEN_PATTERN)),
    ("classifier", MultinomialNB()),
])
sentiment.fit(train_text, train_y)

Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(token_pattern='(?u)\\b[а-яё]+\\b')),
                ('classifier', MultinomialNB())])

In [74]:
probes = ["отличный фильм", "ужасный фильм", "фильм не понравился", "жираф"]
pd.DataFrame({"текст": probes, "прогноз": sentiment.predict(probes)})

,текст,прогноз
0,отличный фильм,pos
1,ужасный фильм,neg
2,фильм не понравился,neg
3,жираф,neg


Посмотрите на последнюю строку: классификатор обычно выдаёт класс даже для нулевого вектора.
В нашем NB тогда решение определяется априорными вероятностями классов;
при равенстве работает правило разрешения ничьей. Такой ответ не означает понимания текста.

## 6. Как оценить предсказания

В бинарной задаче заранее выбираем **положительный класс**. Для отзывов это `pos`,
для фильтра спама ниже — `spam`. «Положительный» в метриках не обязательно значит «хороший».

- **TP**: класс положительный, и модель его нашла.
- **FP**: модель назвала положительным объект другого класса.
- **FN**: модель пропустила положительный объект.
- **TN**: модель правильно определила отрицательный класс.

$$\mathrm{accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\qquad
\mathrm{precision}=\frac{TP}{TP+FP},\qquad
\mathrm{recall}=\frac{TP}{TP+FN}.$$

$$F_1=\frac{2PR}{P+R}.$$

In [75]:
TP, FP, FN, TN = 2, 1, 1, 6
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print("Accuracy:", round((TP + TN) / 10, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(2 * precision * recall / (precision + recall), 3))

Accuracy: 0.8
Precision: 0.667
Recall: 0.667
F1: 0.667


<details>
<summary>После обсуждения: пояснение</summary>

TP = 2, FP = 1, FN = 1, TN = 6. Precision = recall = F1 = 2/3, accuracy = 0.8.
FP может спрятать важное обычное письмо, FN — оставить спам во входящих.
Выбор метрики зависит от последствий этих двух ошибок.

</details>

### Матрица ошибок на отложенных отзывах

В scikit-learn **строки — истинные классы, столбцы — предсказанные**.
Задаём порядок `neg, pos` явно; таблица тогда имеет вид `[[TN, FP], [FN, TP]]`.

In [76]:
from sklearn.metrics import confusion_matrix, classification_report

predicted = sentiment.predict(valid_text)
matrix = confusion_matrix(valid_y, predicted, labels=["neg", "pos"])
pd.DataFrame(matrix, index=["истина neg", "истина pos"],
             columns=["прогноз neg", "прогноз pos"])

,прогноз neg,прогноз pos
истина neg,4,0
истина pos,1,3


In [77]:
print(classification_report(valid_y, predicted, labels=["neg", "pos"],
                            digits=3, zero_division=0))

              precision    recall  f1-score   support

         neg      0.800     1.000     0.889         4
         pos      1.000     0.750     0.857         4

    accuracy                          0.875         8
   macro avg      0.900     0.875     0.873         8
weighted avg      0.900     0.875     0.873         8



`support` — число объектов класса. `macro avg` — среднее метрик по классам с равным весом;
`weighted avg` учитывает число объектов каждого класса.
При нулевом знаменателе `zero_division=0` выводит 0: это соглашение отчёта.

In [78]:
imbalanced_y = ["ham"] * 19 + ["spam"]
always_ham = ["ham"] * 20
print("Accuracy:", accuracy_score(imbalanced_y, always_ham))
print(classification_report(imbalanced_y, always_ham,
                            labels=["ham", "spam"], zero_division=0))

Accuracy: 0.95
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97        19
        spam       0.00      0.00      0.00         1

    accuracy                           0.95        20
   macro avg       0.47      0.50      0.49        20
weighted avg       0.90      0.95      0.93        20



<details>
<summary>После обсуждения: пояснение</summary>

Accuracy = 0.95, но recall и F1 для спама равны 0.
Смотрим метрики по классам; macro F1 тоже покажет проблему — примерно 0.487.
Высокая общая доля правильных ответов не гарантирует полезности модели для редкого класса.

</details>

### Читаем ошибки

Одна ошибка меняет accuracy на нашей проверке сразу на 1/8 = 0.125.

In [79]:
results = pd.DataFrame({"текст": valid_text, "истина": valid_y,
                        "прогноз": predicted})
results[results["истина"] != results["прогноз"]]

,текст,истина,прогноз
7,интересный сюжет и хорошие герои,pos,neg


Для каждой ошибки предложите причину: неизвестное слово, словоформа, отрицание,
недостаток примеров или спорная метка. Если ошибок нет, это повод проверить
размер и сложность выборки

## 7. Сравнение признаков и кросс-валидация

При трёх фолдах обучающая часть трижды делится на две части для обучения и одну для проверки.
Каждый объект один раз попадает в проверочный фолд. Считаем **macro F1**.

`Pipeline` передаём **целиком**: в каждом фолде словарь и IDF обучаются заново
только на его обучающей части. Отложенные 8 отзывов здесь не используем.
Повторные варианты выбирают по CV; для окончательной оценки нужен независимый тест,
который не участвовал в выборе. Наш маленький ноутбук демонстрирует процедуру.

In [80]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
vectorizers = {
    "BoW": CountVectorizer(token_pattern=TOKEN_PATTERN),
    "TF-IDF": TfidfVectorizer(token_pattern=TOKEN_PATTERN),
    "TF-IDF + биграммы": TfidfVectorizer(token_pattern=TOKEN_PATTERN,
                                           ngram_range=(1, 2)),
    "Символы 3–5": TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)),
}

In [81]:
comparison = []
for name, vectorizer in vectorizers.items():
    model = Pipeline([("vectorizer", vectorizer), ("classifier", MultinomialNB())])
    scores = cross_val_score(model, train_text, train_y, cv=cv, scoring="f1_macro")
    comparison.append([name, *scores, scores.mean()])
pd.DataFrame(comparison, columns=["признаки", "фолд 1", "фолд 2", "фолд 3",
                                   "средний F1"]).round(3)

,признаки,фолд 1,фолд 2,фолд 3,средний F1
0,BoW,0.619,0.5,0.750,0.623
1,TF-IDF,0.500,0.5,0.750,0.583
2,TF-IDF + биграммы,0.733,0.5,0.750,0.661
3,Символы 3–5,0.273,0.5,0.619,0.464


<details>
<summary>После обсуждения: пояснение</summary>

Символьные фрагменты могут связать похожие словоформы, словные биграммы — сохранить локальное отрицание.
Это гипотезы, а результат смотрим в таблице. На других данных порядок может измениться.
Сохранив классификатор и разбиения, легче связать изменение метрики именно с признаками.
Если есть дубликаты, отзывы одного автора или сообщения одной цепочки, делить нужно по группам;
если предсказываем будущее — учитывать время. Случайное разбиение подходит не всегда.

</details>

## 8. три классификатора NLTK

NLTK принимает **словарь признаков**, scikit-learn — обычно матрицу.
Сравним интерфейсы на тех же обучающих отзывах.

| Классификатор | Основная идея |
|---|---|
| Naive Bayes | Сопоставляет вероятности признаков в классах с предположением условной независимости |
| Decision Tree | Последовательно проверяет признаки и ветвится по результату |
| MaxEnt | Обучает веса признаков в условной логлинейной модели классов |

Здесь для всех трёх используем **бинарные словные признаки**

In [82]:
from nltk.tokenize import RegexpTokenizer
from nltk.classify import NaiveBayesClassifier, DecisionTreeClassifier, MaxentClassifier

tokenizer = RegexpTokenizer(r"[а-яё]+")
train_word_sets = [set(tokenizer.tokenize(text.lower())) for text in train_text]
nltk_vocabulary = sorted(set().union(*train_word_sets))

In [83]:
def binary_features(text):
    present = set(tokenizer.tokenize(text.lower()))
    return {word: word in present for word in nltk_vocabulary}

nltk_train = [(binary_features(text), label)
              for text, label in zip(train_text, train_y)]

Все признаки имеют значение `True` или `False`, в том числе для отсутствующих слов.
Словарь снова берём только из обучающей части. `RegexpTokenizer` не требует загрузки данных NLTK.

In [84]:
nltk_nb = NaiveBayesClassifier.train(nltk_train)
nltk_tree = DecisionTreeClassifier.train(nltk_train, depth_cutoff=5)

In [85]:
nltk_maxent = MaxentClassifier.train(
    nltk_train, algorithm="iis", max_iter=15, trace=0,
)

In [86]:
nltk_models = {"Naive Bayes": nltk_nb, "Decision Tree": nltk_tree,
               "MaxEnt": nltk_maxent}
pd.DataFrame({name: [model.classify(binary_features(text)) for text in probes]
              for name, model in nltk_models.items()}, index=probes)

,Naive Bayes,Decision Tree,MaxEnt
отличный фильм,pos,pos,pos
ужасный фильм,neg,neg,neg
фильм не понравился,pos,neg,pos
жираф,neg,pos,neg


Классы могут различаться: алгоритмы обучаются по-разному.
В этом примере глубина дерева и число итераций MaxEnt ограничены для короткой демонстрации;
их качество не представляет предел возможностей этих методов.

## 9. тот же конвейер для спама

Меняются тексты и метки, порядок шагов сохраняется.
Обучающие и проверочные сообщения ниже заданы **отдельно заранее**.
`spam` — нежелательное сообщение, `ham` — обычное.
Числа и латиница могли бы быть полезны для спама; здесь сохраняем прежние правила
выделения русских слов, чтобы показать перенос конвейера.

In [87]:
spam_train = [
    "вы выиграли приз заберите подарок", "бесплатный приз только сегодня",
    "срочно получите выигрыш", "подарок каждому участнику акции",
    "сегодня скидка на всё заберите купон", "выгодная акция получите подарок",
]
ham_train = [
    "встреча завтра в библиотеке", "пришли пожалуйста конспект лекции",
    "занятие сегодня в аудитории", "обсудим проект после лекции",
    "завтра встреча с преподавателем", "спасибо за конспект до встречи",
]

In [88]:
spam_model = Pipeline([
    ("vectorizer", TfidfVectorizer(token_pattern=TOKEN_PATTERN)),
    ("classifier", MultinomialNB()),
])
spam_model.fit(spam_train + ham_train, ["spam"] * 6 + ["ham"] * 6)

Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(token_pattern='(?u)\\b[а-яё]+\\b')),
                ('classifier', MultinomialNB())])

### Стоп 11.

Какой класс вы дадите письму «завтра на лекции обсудим спам и призы»?
Обязательно ли слово `приз` означает спам? Что должно измениться в метриках,
если важнее не терять обычные письма?

In [89]:
spam_valid = [
    "получите бесплатный подарок сегодня", "вы выиграли заберите приз",
    "акция срочно получите купон", "пришли конспект завтра встреча",
    "обсудим проект в аудитории", "завтра на лекции обсудим спам и призы",
]
spam_y = ["spam", "spam", "spam", "ham", "ham", "ham"]
spam_pred = spam_model.predict(spam_valid)
pd.DataFrame({"текст": spam_valid, "истина": spam_y, "прогноз": spam_pred})

,текст,истина,прогноз
0,получите бесплатный подарок сегодня,spam,spam
1,вы выиграли заберите приз,spam,spam
2,акция срочно получите купон,spam,spam
3,пришли конспект завтра встреча,ham,ham
4,обсудим проект в аудитории,ham,ham
5,завтра на лекции обсудим спам и призы,ham,ham


In [90]:
print(classification_report(spam_y, spam_pred, labels=["ham", "spam"],
                            digits=3, zero_division=0))

              precision    recall  f1-score   support

         ham      1.000     1.000     1.000         3
        spam      1.000     1.000     1.000         3

    accuracy                          1.000         6
   macro avg      1.000     1.000     1.000         6
weighted avg      1.000     1.000     1.000         6



<details>
<summary>После обсуждения: пояснение</summary>

Письмо о лекции — `ham` по принятой здесь разметке; тема спама ещё не делает сообщение спамом.
Чтобы реже терять обычные письма, нужно снижать число FP для класса `spam`,
то есть контролировать его precision и абсолютное число ложных срабатываний.
Одних ключевых слов мало; нужны разнообразные примеры и проверка на новых сообщениях.
Шесть проверочных сообщений — демонстрация вычисления метрик, а не надёжная оценка фильтра.

</details>

## Документация для повторения

- [scikit-learn: извлечение признаков из текста](https://scikit-learn.org/stable/modules/feature_extraction.html).
- [TfidfVectorizer: параметры, сглаживание IDF и нормировка](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Pipeline: последовательность преобразований и модель](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).
- [Кросс-валидация](https://scikit-learn.org/stable/modules/cross_validation.html).
- [Матрица ошибок: порядок истинных и предсказанных классов](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).
- [NLTK: примеры классификаторов](https://www.nltk.org/howto/classify.html).
